In [ ]:
# from google.colab import drive
# import os

# drive.mount('/content/drive')

Mounted at /content/drive


# Original UCE

In [ ]:
import torch
torch.set_grad_enabled(False)
import argparse
import os
import copy
import time
import pandas as pd

from safetensors.torch import save_file
from diffusers import DiffusionPipeline
import glob

def UCE(pipe, edit_concepts, guide_concepts, preserve_concepts, erase_scale, preserve_scale, lamb, save_dir, exp_name):
    start_time = time.time()
    # Prepare the cross attention weights required to do UCE
    uce_modules = []
    uce_module_names = []
    for name, module in pipe.unet.named_modules():
        if 'attn2' in name and (name.endswith('to_v') or name.endswith('to_k')):
            uce_modules.append(module)
            uce_module_names.append(name)


    original_modules = copy.deepcopy(uce_modules)
    uce_modules = copy.deepcopy(uce_modules)

    # collect text embeddings for erase concept and retain concepts
    uce_erase_embeds = {}
    for e in edit_concepts + guide_concepts + preserve_concepts:
        if e in uce_erase_embeds:
            continue
        t_emb = pipe.encode_prompt(prompt=e,
                                   device=device,
                                   num_images_per_prompt=1,
                                   do_classifier_free_guidance=False)

        last_token_idx = (pipe.tokenizer(e,
                                          padding="max_length",
                                          max_length=pipe.tokenizer.model_max_length,
                                          truncation=True,
                                          return_tensors="pt",
                                         )['attention_mask']).sum()-2


        uce_erase_embeds[e] = t_emb[0][:,last_token_idx,:]

    # collect cross attention outputs for guide concepts and retain concepts (this is for original model weights)
    uce_guide_outputs = {}
    for g in guide_concepts + preserve_concepts:
        if g in uce_guide_outputs:
            continue

        t_emb = uce_erase_embeds[g]

        for module in original_modules:
            uce_guide_outputs[g] = uce_guide_outputs.get(g, []) + [module(t_emb)]

    ###### UCE Algorithm (variables are named according to the paper: https://arxiv.org/abs/2308.14761)
    for module_idx, module in enumerate(original_modules):
        # get original weight of the model
        w_old = module.weight

        # for the left hand term in equation 7 from the paper
        mat1 = lamb * w_old
        # for the right hand term in equation 7 from the paper (we will inverse this later)
        mat2 = lamb * torch.eye(w_old.shape[1], device = device, dtype=torch_dtype)

        # Erase Concepts
        for erase_concept, guide_concept in zip(edit_concepts, guide_concepts):
            c_i = uce_erase_embeds[erase_concept].T
            v_i_star = uce_guide_outputs[guide_concept][module_idx].T

            mat1 += erase_scale * (v_i_star @ c_i.T)
            mat2 += erase_scale * (c_i @ c_i.T)

        # Retain Concepts
        for preserve_concept in preserve_concepts:
            c_i = uce_erase_embeds[preserve_concept].T
            v_i_star = uce_guide_outputs[preserve_concept][module_idx].T

            mat1 += preserve_scale * (v_i_star @ c_i.T)
            mat2 += preserve_scale * (c_i @ c_i.T)


        uce_modules[module_idx].weight = torch.nn.Parameter(mat1 @ torch.inverse(mat2.float()).to(torch_dtype))

    # save the weights
    uce_state_dict = {}
    for name, parameter in zip(uce_module_names, uce_modules):
        uce_state_dict[name+'.weight'] = parameter.weight
        print(name)
        print(parameter)

    file_path = os.path.join(save_dir, exp_name + '.safetensors')
    save_file(uce_state_dict, file_path)
    print("Saved to:", os.path.abspath(file_path))


    end_time = time.time()
    print(f'\n\nErased concepts using UCE\nModel edited in {end_time-start_time} seconds\n')

if __name__ == '__main__':
    import sys

    folder_path = '/content/drive/My Drive/thesis/fgst-de/artists/'
    config_files = glob.glob(os.path.join(folder_path, '*.csv'))

    for config_file in config_files:
        print(f"Processing config: {config_file}")
        df = pd.read_csv(config_file)

        erased_species = df[df["type"] == "erased"]["species"].unique().tolist()
        print("Erased species:", erased_species)

        filename = os.path.basename(config_file)         # gets 'myfile.csv'
        name_only = os.path.splitext(filename)[0]        # removes extension → 'myfile'

        sys.argv = [
            'colab',  # fake script name; doesn't matter
            '--model_id', 'CompVis/stable-diffusion-v1-4',
            '--edit_concepts', ";".join(erased_species),
            '--guide_concepts', '',
            '--preserve_concepts', "",
            '--device', 'cuda:0',
            '--concept_type', 'art',
            '--exp_name', name_only,
        ]

        parser = argparse.ArgumentParser(
                        prog = 'TrainUCE',
                        description = 'UCE for erasing concepts in Stable Diffusion')
        parser.add_argument('--edit_concepts', help='prompts corresponding to concepts to erase separated by ;', type=str, required=True)
        parser.add_argument('--guide_concepts', help='Concepts to guide the erased concepts towards seperated by ;', type=str, default=None)
        parser.add_argument('--preserve_concepts', help='Concepts to preserve seperated by ;', type=str, default=None)
        parser.add_argument('--concept_type', help='type of concept being erased', choices=['art', 'object'], type=str, required=True)

        parser.add_argument('--model_id', help='Model to run UCE on', type=str, default="CompVis/stable-diffusion-v1-4",)
        parser.add_argument('--device', help='cuda devices to train on', type=str, required=False, default='cuda:0')

        parser.add_argument('--erase_scale', help='scale to erase concepts', type=float, required=False, default=1)
        parser.add_argument('--preserve_scale', help='scale to preserve concepts', type=float, required=False, default=1)
        parser.add_argument('--lamb', help='lambda regularization term for UCE', type=float, required=False, default=0.5)

        parser.add_argument('--expand_prompts', help='do you wish to expand your prompts?', choices=['true', 'false'], type=str, required=False, default='false')

        parser.add_argument('--save_dir', help='where to save your uce model weights', type=str, default='uce_models')
        parser.add_argument('--exp_name', help='Use this to name your saved filename', type=str, default=None)

        args = parser.parse_args()

        device = args.device
        torch_dtype = torch.float32
        model_id = args.model_id

        preserve_scale = args.preserve_scale
        erase_scale = args.erase_scale
        lamb = args.lamb

        concept_type = args.concept_type
        expand_prompts = args.expand_prompts

        save_dir = args.save_dir
        os.makedirs(save_dir, exist_ok=True)
        exp_name = args.exp_name
        if exp_name is None:
            exp_name = 'uce_test'

        # erase concepts
        edit_concepts = [concept.strip() for concept in args.edit_concepts.split(';')]
        # guide concepts
        guide_concepts = args.guide_concepts
        if guide_concepts is None:
            guide_concepts = ''
            if concept_type == 'art':
                guide_concepts = 'art'
        guide_concepts = [concept.strip() for concept in guide_concepts.split(';')]
        if len(guide_concepts) == 1:
            guide_concepts = guide_concepts*len(edit_concepts)
        if len(guide_concepts) != len(edit_concepts):
            raise Exception('Error! The length of erase concepts and their corresponding guide concepts do not match. Please make sure they are seperated by ; and are of equal sizes')

        # preserve concepts
        if args.preserve_concepts is None:
            preserve_concepts = []
        else:
            preserve_concepts = [concept.strip() for concept in args.preserve_concepts.split(';')]



        if expand_prompts == 'true':
            edit_concepts_ = copy.deepcopy(edit_concepts)
            guide_concepts_ = copy.deepcopy(guide_concepts)

            for concept, guide_concept in zip(edit_concepts_, guide_concepts_):
                if concept_type == 'art':
                    edit_concepts.extend([f'painting by {concept}',
                                          f'art by {concept}',
                                          f'artwork by {concept}',
                                          f'picture by {concept}',
                                          f'style of {concept}'
                                          ]
                                        )
                    guide_concepts.extend([f'painting by {guide_concept}',
                                          f'art by {guide_concept}',
                                          f'artwork by {guide_concept}',
                                          f'picture by {guide_concept}',
                                          f'style of {guide_concept}'
                                          ]
                                        )

                else:
                    edit_concepts.extend([f'image of {concept}',
                                          f'photo of {concept}',
                                          f'portrait of {concept}',
                                          f'picture of {concept}',
                                          f'painting of {concept}'
                                          ]
                                        )
                    guide_concepts.extend([f'image of {guide_concept}',
                                          f'photo of {guide_concept}',
                                          f'portrait of {guide_concept}',
                                          f'picture of {guide_concept}',
                                          f'painting of {guide_concept}'
                                          ]
                                        )


        print(f"\n\nErasing: {edit_concepts}\n")
        print(f"Guiding: {guide_concepts}\n")
        print(f"Preserving: {preserve_concepts}\n")

        pipe = DiffusionPipeline.from_pretrained(model_id,
                                                torch_dtype=torch_dtype,
                                                safety_checker=None,
                                                vae=None).to(device)

        UCE(pipe, edit_concepts, guide_concepts, preserve_concepts, erase_scale, preserve_scale, lamb, save_dir, exp_name)

Test
['/content/drive/My Drive/thesis/fgst-de/artists/lace_generate_van-gogh.csv', '/content/drive/My Drive/thesis/fgst-de/artists/lace_generate_kelly-mckernan.csv']
Processing config: /content/drive/My Drive/thesis/fgst-de/artists/lace_generate_van-gogh.csv
Erased species: ['Van Gogh']


Erasing: ['Van Gogh']

Guiding: ['']

Preserving: ['']



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

scheduler_config-checkpoint.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_k
Linear(in_features=768, out_features=320, bias=False)
down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_v
Linear(in_features=768, out_features=320, bias=False)
down_blocks.0.attentions.1.transformer_blocks.0.attn2.to_k
Linear(in_features=768, out_features=320, bias=False)
down_blocks.0.attentions.1.transformer_blocks.0.attn2.to_v
Linear(in_features=768, out_features=320, bias=False)
down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_k
Linear(in_features=768, out_features=640, bias=False)
down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_v
Linear(in_features=768, out_features=640, bias=False)
down_blocks.1.attentions.1.transformer_blocks.0.attn2.to_k
Linear(in_features=768, out_features=640, bias=False)
down_blocks.1.attentions.1.transformer_blocks.0.attn2.to_v
Linear(in_features=768, out_features=640, bias=False)
down_blocks.2.attentions.0.transformer_blocks.0.attn2.to_k
Linear(in_features=768, out_features=

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_k
Linear(in_features=768, out_features=320, bias=False)
down_blocks.0.attentions.0.transformer_blocks.0.attn2.to_v
Linear(in_features=768, out_features=320, bias=False)
down_blocks.0.attentions.1.transformer_blocks.0.attn2.to_k
Linear(in_features=768, out_features=320, bias=False)
down_blocks.0.attentions.1.transformer_blocks.0.attn2.to_v
Linear(in_features=768, out_features=320, bias=False)
down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_k
Linear(in_features=768, out_features=640, bias=False)
down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_v
Linear(in_features=768, out_features=640, bias=False)
down_blocks.1.attentions.1.transformer_blocks.0.attn2.to_k
Linear(in_features=768, out_features=640, bias=False)
down_blocks.1.attentions.1.transformer_blocks.0.attn2.to_v
Linear(in_features=768, out_features=640, bias=False)
down_blocks.2.attentions.0.transformer_blocks.0.attn2.to_k
Linear(in_features=768, out_features=

In [ ]:
from diffusers import DiffusionPipeline
import torch
from PIL import Image
import pandas as pd
import argparse
import os
torch.set_grad_enabled(False)
from safetensors.torch import load_file
import glob
from tqdm import tqdm

def generate_images(model_id, uce_model_path, prompts_path, save_path, exp_name='test', device='cuda:0', torch_dtype=torch.bfloat16, guidance_scale = 7.5, num_inference_steps=100, num_images_per_prompt=10, from_case=0, till_case=1000000):

    # 1. Load the pipe
    pipe = DiffusionPipeline.from_pretrained(model_id,
                                         torch_dtype=torch_dtype,
                                         safety_checker=None).to(device)

    if uce_model_path is not None:
        uce_weights = load_file(uce_model_path)
        pipe.unet.load_state_dict(uce_weights, strict=False)

    df = pd.read_csv(prompts_path)

    folder_path = f'{save_path}/{exp_name}'
    os.makedirs(folder_path, exist_ok=True)

    for index, row in tqdm(df.iterrows()):
        prompt = str(row.prompt)
        seed = row.evaluation_seed
        species = row.species
        concept_type = row["type"]

        pil_images = pipe(prompt=prompt,
                          num_inference_steps=num_inference_steps,
                          guidance_scale=guidance_scale,
                          num_images_per_prompt=num_images_per_prompt,
                          generator=torch.Generator().manual_seed(seed)
                         ).images


        for num, im in enumerate(pil_images):
            safe_species = species.replace(" ", "_").replace("/", "_").replace("-", "_")
            print("Saving to:" + f"{folder_path}/uce_{safe_species}_{index:04d}_seed{seed}_{concept_type}_im{num}.png")
            im.save(f"{folder_path}/uce_{safe_species}_{index:04d}_seed{seed}_{concept_type}_im{num}.png")

if __name__=='__main__':
    import sys

    folder_path = '/content/drive/My Drive/thesis/fgst-de/artists/'
    config_files = glob.glob(os.path.join(folder_path, '*.csv'))

    for config_file in config_files:
        print(f"Processing config: {config_file}")
        df = pd.read_csv(config_file)

        erased_species = df[df["type"] == "erased"]["species"].unique().tolist()
        print("Erased species:", erased_species)

        filename = os.path.basename(config_file)         # gets 'myfile.csv'
        name_only = os.path.splitext(filename)[0]        # removes extension → 'myfile'

        uce_path = f'/content/drive/MyDrive/thesis/unified-concept-editing/uce_models/{name_only}.safetensors'

        sys.argv = [
            'colab',  # fake script name; doesn't matter
            '--model_id', 'CompVis/stable-diffusion-v1-4',
            '--uce_model_path', uce_path,
            '--prompts_path', config_file,
            '--device', 'cuda:0',
            '--exp_name', name_only,
        ]


        parser = argparse.ArgumentParser(
                        prog = 'generateImages',
                        description = 'Generate Images using Diffusers Code')
        parser.add_argument('--model_id', help='hf repo id for the model you want to test', type=str, required=False, default='CompVis/stable-diffusion-v1-4')
        parser.add_argument('--uce_model_path', help='path for uce model', type=str, required=False, default=None)
        parser.add_argument('--prompts_path', help='path to csv file with prompts', type=str, required=True)
        parser.add_argument('--save_path', help='folder where to save images', type=str, required=False, default='uce_results')
        parser.add_argument('--device', help='cuda device to run on', type=str, required=False, default='cuda:0')
        parser.add_argument('--exp_name', help='foldername to save the results', type=str, required=False, default='test_images')
        parser.add_argument('--guidance_scale', help='guidance to run eval', type=float, required=False, default=7.5)
        parser.add_argument('--till_case', help='continue generating from case_number', type=int, required=False, default=1000000)
        parser.add_argument('--from_case', help='continue generating from case_number', type=int, required=False, default=0)
        parser.add_argument('--num_images_per_prompt', help='number of samples per prompt', type=int, required=False, default=1)
        parser.add_argument('--num_inference_steps', help='ddim steps of inference used to train', type=int, required=False, default=50)
        args = parser.parse_args()

        model_id = args.model_id
        uce_model_path = args.uce_model_path
        prompts_path = args.prompts_path
        save_path = args.save_path
        device = args.device
        guidance_scale = args.guidance_scale
        exp_name = args.exp_name
        num_images_per_prompt= args.num_images_per_prompt
        from_case = args.from_case
        till_case = args.till_case
        num_inference_steps = args.num_inference_steps
        generate_images(model_id=model_id, uce_model_path=uce_model_path, prompts_path=prompts_path, save_path=save_path, exp_name=exp_name, device=device, torch_dtype=torch.bfloat16, guidance_scale = guidance_scale, num_inference_steps=num_inference_steps, num_images_per_prompt=num_images_per_prompt, from_case=from_case, till_case=till_case)

Processing config: /content/drive/My Drive/thesis/fgst-de/artists/lace_generate_van-gogh.csv
Erased species: ['Van Gogh']


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
0it [00:00, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

1it [00:04,  4.67s/it]

...

99it [05:35,  3.41s/it]

Saving to:uce_results/lace_generate_kelly-mckernan/uce_Ajin:_Demi_Human_0098_seed2242_other_im0.png


  0%|          | 0/50 [00:00<?, ?it/s]

100it [05:38,  3.39s/it]

Saving to:uce_results/lace_generate_kelly-mckernan/uce_Ajin:_Demi_Human_0099_seed3590_other_im0.png
